# Protein Prediction Model

## Setup

### Imports

In [35]:
import tensorflow as tf
import keras
import numpy as np
import matplotlib.pyplot as plt
import os, glob, pathlib
import pandas as pd
import datetime as dt
import time

### Data Aquisition

Data needs to be imported from its raw .xls format then processed into separate dataframes for each protein's item.

In [ ]:
# TODO: Pull in all .xls files in the data directory and process them into their respective dictionaries of dataframes.
data_path = pathlib.Path.cwd() / "data" / "03_16_03_21_Strips.xls"  # Path to the Excel file containing the sales data
df = pd.read_excel(data_path)

df = df.dropna(how='all')   # Drop rows where all values are NaN
df = df.iloc[6:, :]         # Drop the first 6 rows which contain metadata and are not part of the actual data
df = df[df.iloc[:, 0] != "Sales Mix Time Interval Report"]  # Drop rows where the first column contains "Sales Mix Time Interval Report"
df = df[df.iloc[:, 0] != "Store: Indian Trail FSU, 02965"]  # Drop rows where the first column contains "Store: Indian Trail FSU, 02965"
df = df[df.iloc[:, 3] != "Totals:"] # Drop rows where the fourth column contains "Totals:"
df = df.dropna(axis=1, how='all')   # Drop columns where all values are NaN (holdovers from PDF formatting)
df = df.drop(df.columns[1], axis=1) # Drop the second column containing only the word "Time"
df.reset_index(drop=True, inplace=True) # Reset the index of the dataframe after dropping rows


def convert_to_24h(time_str: str) -> dt.time:
    '''
    Convert a time string in the format "HH:MM AM/PM - HH:MM AM/PM" 
    to a datetime.time object in 24-hour format.

    Args:
        time_str (str): The time string to convert.

    Returns:
        dt.time: The start time in 24-hour format.
    '''
    time_components = time_str.split()  # Split the time string into components by whitespace
    start_time = time_components[0]     # The first component is the start time (e.g., "10:00")
    am_pm = time_components[3]          # The fourth component is the AM/PM indicator for the end time (e.g., "AM" or "PM")
    hours, minutes = map(int, start_time.split(':'))    # Split the start time into hours and minutes and convert to integers
    if am_pm == 'PM' and hours != 12:       # If the time is in PM and not 12 PM, add 12 to convert to 24-hour format
        hours += 12
    elif am_pm == 'AM' and hours == 12:     # If the time is 12 AM, set hours to 0 to convert to 24-hour format
        hours = 0
    return dt.time(hour=hours, minute=minutes)  # Return the start time as a datetime.time object in 24-hour format


# TODO: Map time intervals to integers to facilitate model training, and store the mapping in a dictionary for later use in prediction.
# Pull start times from the time interval column and convert to 24-hour format.
# df['Start Time'] = df.iloc[:, 1].map(lambda x: convert_to_24h(x) if pd.notnull(x) else np.nan)
# Format the dataframe to store pertinent information such as day-of-week and item name in the metadata.
df.rename(columns={df.columns[0]: 'Item', df.columns[1]: 'Time Interval', df.columns[2]: 'Monday',
                   df.columns[3]: 'Tuesday', df.columns[4]: 'Wednesday', df.columns[5]: 'Thursday',
                   df.columns[6]: 'Friday', df.columns[7]: 'Saturday'}, inplace=True)
for col in df.columns[2:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')   # Convert the sales data columns to numeric, coercing errors to NaN


df_dict : dict[str, pd.DataFrame] = {}  # Initialize an empty dictionary to store the dataframes for each item
for item in df['Item'].unique():        # Only create dataframes for unique, non-null item names in the 'Item' column
    if pd.notnull(item):
        df_dict[item] = df[df['Item'] == item].copy()
name_idx_start = 0
name_idx_end = 0
for index, row in df.iterrows():        # Grab each section of the dataframe corresponding to each item and store it in the dictionary with the item name as the key
    if pd.notnull(row['Item']):
        if name_idx_end != 0:           # First iteration guard
            df_dict[str(df.iloc[name_idx_start, 0])] = df.iloc[name_idx_start:name_idx_end, :].copy()
        name_idx_start = name_idx_end       # Move the start index to the current end index for the next item section
        name_idx_end = name_idx_start + 1   # Move the end index to the next row to start looking for the next item name
    elif index == len(df) - 1:          # If we've reached the end of the dataframe, store the last item section in the dictionary
        df_dict[str(df.iloc[name_idx_start, 0])] = df.iloc[name_idx_start:name_idx_end + 1, :].copy()
    else:                               # If the current row does not contain an item name, move the end index down to continue looking for the next item name
        name_idx_end += 1
        continue


for key in df_dict.keys():  # Adjust each dataframe to keep only the relevant data.
    df_dict[key].drop(df_dict[key].index[:2], inplace=True)             # Item name and days of the week are stored in datafram metadata
    df_dict[key].drop(df_dict[key].columns[0], axis=1, inplace=True)    # Drop the 'Item' column as it is redundant after storing the item name in the dictionary key
    df_dict[key].reset_index(drop=True, inplace=True)                   # Reset the index of the dataframe after dropping rows and columns
    #for col in df_dict[key].columns[1:-1]:                             # Convert the sales data columns to numeric, coercing errors to NaN (holdovers from PDF formatting)
        #df_dict[key][col] = pd.to_numeric(df_dict[key][col], errors='coerce')
    # Above loop is redundant after converting the entire dataframe to numeric earlier, but may be useful for future dataframes that are not preprocessed in the same way.

WARNING *** file size (90403) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero


,Time Interval,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday
0,10:15 - 10:29 AM,NaN,NaN,NaN,NaN,1.0,NaN
1,10:30 - 10:44 AM,1.0,1.0,4.0,NaN,2.0,NaN
2,10:45 - 10:59 AM,1.0,4.0,NaN,1.0,2.0,1.0
3,11:00 - 11:14 AM,NaN,2.0,NaN,NaN,NaN,NaN
4,11:15 - 11:29 AM,NaN,3.0,3.0,2.0,NaN,3.0
5,11:30 - 11:44 AM,1.0,1.0,NaN,5.0,2.0,2.0
6,11:45 - 11:59 AM,1.0,2.0,1.0,3.0,1.0,1.0
7,12:00 - 12:14 PM,2.0,1.0,4.0,6.0,3.0,1.0
8,12:15 - 12:29 PM,4.0,1.0,3.0,3.0,4.0,3.0
9,12:30 - 12:44 PM,NaN,5.0,2.0,6.0,4.0,NaN
